# Local notebook -> Modal function calling

This is the complete Modal workflow: prepare the local project, authenticate, start the local notebook kernel, define a remote GPU function, call it, and let Modal release the function container.

## Before opening Cursor

Run this on the Mac:

```bash
cd ~/Developer/ml-lab
uv sync --extra remote
uv run modal setup
```

`modal setup` authenticates the Modal CLI. The final function call is billable. Never commit Modal credentials.

## Start the local kernel

1. Open the project with `cursor -r .`.
2. Open this notebook locally from the Explorer.
3. Click the current kernel button, or `Select Kernel` on first use.
4. Choose `Select Another Kernel...` -> `Python Environments...` -> `.venv/bin/python`.
5. Run the first cell. This starts the local kernel and loads the Modal client.

## Keyboard controls on macOS

- Execute the selected cell: `Control+Enter` (`⌃⏎`).
- Add a code cell: `Command+Enter` (`⌘⏎`).
- Execute everything: click `Run All` in the notebook toolbar.

![Cursor notebook kernel picker flow](../docs/images/cursor-kernel-picker-flow.svg)

The local notebook kernel is only the client. The decorated function runs in a separate Modal container.

## Run the remote function

1. Run the second cell to define `remote_matmul` locally.
2. Run the final cell to call it on a Modal T4 GPU.
3. Confirm the result reports a CUDA device, GPU name, and tensor shape.

## Stop

The function call returns and Modal manages the remote container lifecycle. Restart or close the local notebook kernel when finished.

In [ ]:
import sys

import modal

app = modal.App("ml-lab-notebook")
image = modal.Image.debian_slim(python_version="3.12").pip_install("torch")
print({"python": sys.executable, "modal": modal.__version__, "client": "local"})

In [ ]:
@app.function(image=image, gpu="T4", timeout=300)
def remote_matmul(size: int = 1024) -> dict[str, object]:
    import torch

    left = torch.randn(size, size, device="cuda")
    right = torch.randn(size, size, device="cuda")
    result = left @ right
    return {
        "device": str(result.device),
        "gpu": torch.cuda.get_device_name(0),
        "shape": list(result.shape),
    }


print("Remote function defined locally")

In [ ]:
with modal.enable_output(), app.run():
    result = remote_matmul.remote(1024)

print(result)